# 神经推荐从零实现：候选打分与已购过滤

## 面试问题

面试时我会把推荐拆成召回、排序和约束三层。神经排序模型不应只背用户 ID，还要吸收可迁移的商品侧特征；训练样本中的负例必须代表真实候选，而不能把未来正例误标为负例。离线评估必须让基线和模型面对同一候选集合，并按用户计算 HitRate 或 NDCG。模型分数不是最终推荐，已购、库存、合规和多样性过滤必须在输出前执行。冷启动用户还需要热门或内容特征回退。下面手写一个融合用户 embedding 与商品类目特征的 MLP 排序器，展示真实梯度、候选 logits 和已购泄漏修复。

## 真实案例

数据模拟六名用户在电商站内的脱敏行为，每人有两个历史购买和一个留出的下一次购买；九件商品分属数码、运动、家居三类。训练只把跨类商品作为明确负例，绝不把留出的未来购买当负例。该规则是教学简化，真实系统会使用曝光未点、停留和时间窗。

本实验是为了看清机制而构造的离线小样本，不代表线上收益，也不能外推到开放分布。

In [1]:
import torch  # 导入 PyTorch 以实现神经推荐模型和梯度更新。
from torch import nn  # 导入神经网络层基类。
import torch.nn.functional as F  # 导入二元交叉熵与激活函数。
torch.manual_seed(28)  # 固定随机种子以复现实验输出。
torch.set_num_threads(1)  # 限制线程数以稳定小规模实验运行时间。
users = ["小林", "阿杰", "小周", "晓雨", "陈姨", "老郑"]  # 定义六名脱敏用户。
items = ["手机", "机械键盘", "降噪耳机", "跑鞋", "瑜伽垫", "哑铃", "咖啡机", "台灯", "空气净化器"]  # 定义九件有业务语义的商品。
categories = ["数码", "数码", "数码", "运动", "运动", "运动", "家居", "家居", "家居"]  # 保存每件商品所属类目。
item_features = torch.tensor([[1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 1.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [0.0, 0.0, 1.0], [0.0, 0.0, 1.0]], dtype=torch.float32)  # 用三维 one-hot 表示商品类目。
histories = {0: [0, 1], 1: [1, 2], 2: [3, 4], 3: [4, 5], 4: [6, 7], 5: [7, 8]}  # 保存每位用户的两个训练期购买。
held_out = {0: 2, 1: 0, 2: 5, 3: 3, 4: 8, 5: 6}  # 保存每位用户留出的下一次真实购买。
print("用户  历史购买                 留出购买  偏好类目")  # 打印真实输入预览的表头。
for user_index, user_name in enumerate(users):  # 逐用户展示训练历史和评估目标。
    history_names = "、".join(items[item_index] for item_index in histories[user_index])  # 把历史商品编号转成可读名称。
    preferred_category = categories[histories[user_index][0]]  # 从历史购买读取当前用户偏好类目。
    print(f"{user_name:<3}  {history_names:<20}  {items[held_out[user_index]]:<8}  {preferred_category}")  # 输出当前用户的完整离线记录。
print(f"样本规模：用户={len(users)}，商品={len(items)}，训练正反馈={sum(len(value) for value in histories.values())}")  # 汇总推荐案例规模。

用户  历史购买                 留出购买  偏好类目
小林   手机、机械键盘               降噪耳机      数码
阿杰   机械键盘、降噪耳机             手机        数码
小周   跑鞋、瑜伽垫                哑铃        运动
晓雨   瑜伽垫、哑铃                跑鞋        运动
陈姨   咖啡机、台灯                空气净化器     家居
老郑   台灯、空气净化器              咖啡机       家居
样本规模：用户=6，商品=9，训练正反馈=12


## 基线：训练期全站热门

热门基线统计训练购买次数，再从每个用户未买过的商品里选最热门商品。它简单、可解释，也是检验复杂模型是否真的有价值的必要对照。

In [2]:
popularity = torch.zeros(len(items), dtype=torch.float32)  # 创建每件商品的训练期热度计数。
for bought_items in histories.values():  # 遍历所有用户的历史购买列表。
    for item_index in bought_items:  # 遍历当前用户的每件已购商品。
        popularity[item_index] += 1.0  # 累加对应商品的购买次数。
baseline_hits = []  # 保存每位用户的热门基线命中结果。
baseline_choices = []  # 保存每位用户的热门推荐商品编号。
print("用户  热门推荐  留出商品  是否命中")  # 打印热门基线逐用户结果表头。
for user_index, user_name in enumerate(users):  # 逐用户构造相同的未购候选集合。
    candidate_indices = [item_index for item_index in range(len(items)) if item_index not in histories[user_index]]  # 排除训练期已购商品形成候选集合。
    baseline_choice = max(candidate_indices, key=lambda item_index: (popularity[item_index].item(), -item_index))  # 按热度和稳定编号选择一个候选。
    baseline_hit = int(baseline_choice == held_out[user_index])  # 判断热门推荐是否命中留出购买。
    baseline_choices.append(baseline_choice)  # 保存当前用户的热门推荐编号。
    baseline_hits.append(baseline_hit)  # 保存当前用户的命中标记。
    print(f"{user_name:<3}  {items[baseline_choice]:<8}  {items[held_out[user_index]]:<8}  {baseline_hit}")  # 输出当前用户的推荐对照。
baseline_hit_rate = sum(baseline_hits) / len(baseline_hits)  # 计算热门基线的 HitRate@1。
print(f"热门基线 HitRate@1={baseline_hit_rate:.1%}，商品热度={popularity.tolist()}")  # 汇总同口径基线指标与热度分布。

用户  热门推荐  留出商品  是否命中
小林   瑜伽垫       降噪耳机      0
阿杰   瑜伽垫       手机        0
小周   机械键盘      哑铃        0
晓雨   机械键盘      跑鞋        0
陈姨   机械键盘      空气净化器     0
老郑   机械键盘      咖啡机       0
热门基线 HitRate@1=0.0%，商品热度=[1.0, 2.0, 1.0, 1.0, 2.0, 1.0, 1.0, 2.0, 1.0]


## 手写核心：用户向量、商品侧特征与 MLP 打分

商品塔不用商品 ID embedding，而是投影类目 one-hot，因此留出商品即使没被该用户买过，也能继承同类商品的可迁移表示。训练集只加入跨类明确负例，避免把下一次购买错误地当作负例。

In [3]:
class NeuralRanker(nn.Module):  # 定义融合用户 embedding 与商品类目特征的排序器。
    def __init__(self, user_count, item_feature_dim):  # 根据用户数和商品特征维度创建网络。
        super().__init__()  # 初始化父类以注册所有参数。
        self.user_embedding = nn.Embedding(user_count, 4)  # 为每位用户学习四维偏好向量。
        self.item_projection = nn.Linear(item_feature_dim, 4, bias=False)  # 把类目特征投影到四维商品空间。
        self.hidden_layer = nn.Linear(8, 8)  # 融合用户与商品表示并学习非线性交互。
        self.output_layer = nn.Linear(8, 1)  # 把隐藏表示映射为未归一化相关性 logit。
    def forward(self, user_indices, item_side_features):  # 计算一批用户商品对的相关性分数。
        user_vectors = self.user_embedding(user_indices)  # 查找当前批次的用户偏好向量。
        item_vectors = self.item_projection(item_side_features)  # 投影当前商品的可迁移侧信息。
        joint_vectors = torch.cat([user_vectors, item_vectors], dim=1)  # 拼接用户和商品表示形成交互输入。
        hidden = torch.tanh(self.hidden_layer(joint_vectors))  # 用双向有界激活保留正负方向的偏好匹配证据。
        logits = self.output_layer(hidden).squeeze(1)  # 输出每个用户商品对的单一 logit。
        return logits, hidden  # 返回候选 logits 与可观察的中间隐藏表示。
training_users = []  # 收集点式训练样本中的用户编号。
training_items = []  # 收集点式训练样本中的商品编号。
training_labels = []  # 收集正负反馈标签。
for user_index in range(len(users)):  # 逐用户构造正例和明确跨类负例。
    preferred_category = categories[histories[user_index][0]]  # 读取当前用户由历史行为体现的偏好类目。
    negative_indices = [item_index for item_index, category in enumerate(categories) if category != preferred_category]  # 覆盖另外两个类目的商品以避免未训练类目分数漂移。
    for item_index in histories[user_index]:  # 遍历当前用户的两个训练期正反馈。
        training_users.append(user_index)  # 保存正样本的用户编号。
        training_items.append(item_index)  # 保存正样本的商品编号。
        training_labels.append(1.0)  # 把历史购买标记为正反馈。
    for item_index in negative_indices:  # 遍历当前用户的全部跨类明确负例。
        training_users.append(user_index)  # 保存负样本的用户编号。
        training_items.append(item_index)  # 保存负样本的商品编号。
        training_labels.append(0.0)  # 把跨类商品标记为负反馈。
training_user_tensor = torch.tensor(training_users, dtype=torch.long)  # 转换训练用户编号为张量。
training_item_tensor = torch.tensor(training_items, dtype=torch.long)  # 转换训练商品编号为张量。
training_label_tensor = torch.tensor(training_labels, dtype=torch.float32)  # 转换二元反馈标签为张量。
ranker = NeuralRanker(len(users), item_features.shape[1])  # 实例化手写神经排序器。
print(ranker)  # 展示排序器的真实网络层次。
print(f"训练用户商品对={len(training_labels)}，其中正例={int(training_label_tensor.sum().item())}，留出商品未进入负例")  # 输出训练样本构成。

NeuralRanker(
  (user_embedding): Embedding(6, 4)
  (item_projection): Linear(in_features=3, out_features=4, bias=False)
  (hidden_layer): Linear(in_features=8, out_features=8, bias=True)
  (output_layer): Linear(in_features=8, out_features=1, bias=True)
)
训练用户商品对=48，其中正例=12，留出商品未进入负例


In [4]:
optimizer = torch.optim.Adam(ranker.parameters(), lr=0.05)  # 创建优化器更新用户和商品匹配参数。
loss_trace = []  # 保存真实训练损失轨迹。
first_gradient_norm = 0.0  # 预留首轮用户 embedding 梯度范数。
for epoch in range(251):  # 在小型离线样本上执行二百五十一次更新。
    optimizer.zero_grad()  # 清除上一轮累计梯度。
    training_logits, training_hidden = ranker(training_user_tensor, item_features[training_item_tensor])  # 前向计算所有训练用户商品对。
    loss = F.binary_cross_entropy_with_logits(training_logits, training_label_tensor)  # 计算点式二元交叉熵损失。
    loss.backward()  # 反向传播到用户 embedding 和商品投影层。
    if epoch == 0:  # 只在第一轮记录实际梯度规模。
        first_gradient_norm = ranker.user_embedding.weight.grad.norm().item()  # 读取用户向量的首轮梯度范数。
    optimizer.step()  # 根据当前梯度更新排序模型参数。
    loss_trace.append(loss.item())  # 保存当前轮损失用于观察收敛。
    if epoch in [0, 25, 100, 250]:  # 选择关键轮次展示训练轨迹。
        training_accuracy = ((torch.sigmoid(training_logits) >= 0.5).float() == training_label_tensor).float().mean().item()  # 计算当前训练对分类准确率。
        print(f"epoch={epoch:03d} loss={loss.item():.4f} pair_accuracy={training_accuracy:.1%}")  # 输出真实损失与样本对准确率。
ranker.eval()  # 切换到评估模式生成候选排名。
print(f"首轮用户向量梯度范数={first_gradient_norm:.6f}")  # 证明训练确实经过反向传播更新。

epoch=000 loss=0.7245 pair_accuracy=50.0%
epoch=025 loss=0.4829 pair_accuracy=79.2%


epoch=100 loss=0.0080 pair_accuracy=100.0%


epoch=250 loss=0.0021 pair_accuracy=100.0%
首轮用户向量梯度范数=0.039240


## 结果解读：逐用户候选 logits

每位用户都在“排除历史购买后的全部商品”中排序。表中同时展示前三名候选及原始 logits，便于判断模型为何命中，而不是只看一个 HitRate 数字。

In [5]:
neural_hits = []  # 保存每位用户的神经排序命中标记。
neural_choices = []  # 保存每位用户过滤后的首选商品。
candidate_tables = []  # 保存逐用户候选排序以供后续失败案例复用。
with torch.no_grad():  # 关闭评估阶段的梯度记录。
    for user_index, user_name in enumerate(users):  # 逐用户对全部商品计算可比较分数。
        repeated_users = torch.full((len(items),), user_index, dtype=torch.long)  # 创建与商品数相同的当前用户编号。
        all_logits, all_hidden = ranker(repeated_users, item_features)  # 计算当前用户面对九件商品的 logits 与隐藏表示。
        candidate_indices = [item_index for item_index in range(len(items)) if item_index not in histories[user_index]]  # 排除训练期已购商品形成真实候选。
        ranked_candidates = sorted(candidate_indices, key=lambda item_index: (-all_logits[item_index].item(), item_index))  # 按 logit 从高到低稳定排序候选。
        neural_choice = ranked_candidates[0]  # 选择过滤后得分最高的商品。
        neural_hit = int(neural_choice == held_out[user_index])  # 判断首位推荐是否命中留出购买。
        top_three = [(items[item_index], round(all_logits[item_index].item(), 3)) for item_index in ranked_candidates[:3]]  # 生成可读的前三候选名称与 logits。
        neural_choices.append(neural_choice)  # 保存当前用户的最终推荐商品。
        neural_hits.append(neural_hit)  # 保存当前用户的命中结果。
        candidate_tables.append((all_logits.clone(), all_hidden.clone(), ranked_candidates))  # 保存候选中间量供失败分析。
        print(f"{user_name}: top3={top_three}，留出={items[held_out[user_index]]}，命中={neural_hit}")  # 输出逐用户候选排名和真实目标。
neural_hit_rate = sum(neural_hits) / len(neural_hits)  # 计算神经排序器的 HitRate@1。
example_hidden = candidate_tables[0][1][held_out[0]].tolist()  # 读取小林留出商品的完整八维隐藏表示。
print(f"同候选 HitRate@1：热门={baseline_hit_rate:.1%}，神经排序={neural_hit_rate:.1%}")  # 汇总同一数据和指标下的方案差异。
print(f"小林-降噪耳机的完整隐藏表示={[round(value, 3) for value in example_hidden]}")  # 展示候选 logit 之前的全部中间神经表示。

小林: top3=[('降噪耳机', 5.201), ('咖啡机', -11.565), ('台灯', -11.565)]，留出=降噪耳机，命中=1
阿杰: top3=[('手机', 5.203), ('空气净化器', -12.04), ('咖啡机', -12.04)]，留出=手机，命中=1
小周: top3=[('哑铃', 6.739), ('咖啡机', -6.11), ('台灯', -6.11)]，留出=哑铃，命中=1
晓雨: top3=[('跑鞋', 6.684), ('空气净化器', -6.125), ('咖啡机', -6.125)]，留出=跑鞋，命中=1
陈姨: top3=[('空气净化器', 5.016), ('手机', -6.112), ('机械键盘', -6.112)]，留出=空气净化器，命中=1
老郑: top3=[('咖啡机', 5.022), ('手机', -6.122), ('机械键盘', -6.122)]，留出=咖啡机，命中=1
同候选 HitRate@1：热门=0.0%，神经排序=100.0%
小林-降噪耳机的完整隐藏表示=[1.0, -0.999, -0.999, 1.0, -0.999, 1.0, -1.0, 1.0]


## 失败案例：直接取全量最高分会推荐已购商品

模型训练时会给历史正例高分，因此“模型打分完直接 argmax”通常把已经买过的商品再次推荐。修复方式是在业务输出层应用已购集合过滤；这里用小林的真实候选展示错误与修复。

In [6]:
example_user = 0  # 选择小林作为已购泄漏的可复现案例。
example_logits = candidate_tables[example_user][0]  # 读取小林对全部商品的模型 logits。
raw_top_index = int(torch.argmax(example_logits).item())  # 模拟遗漏已购过滤时直接选择全量最高分。
filtered_top_index = neural_choices[example_user]  # 读取经过已购过滤后的最高分商品。
raw_is_consumed = raw_top_index in histories[example_user]  # 检查错误推荐是否已存在于历史购买。
print(f"未过滤：推荐={items[raw_top_index]}，logit={example_logits[raw_top_index]:.3f}，是否已购={raw_is_consumed}")  # 展示直接 argmax 的业务错误。
print(f"已过滤：推荐={items[filtered_top_index]}，logit={example_logits[filtered_top_index]:.3f}，真实下次购买={items[held_out[example_user]]}")  # 展示过滤后的正确候选。
print("修复结论：模型负责相关性，候选层仍必须执行已购、库存与合规过滤。")  # 总结模型分数与业务约束的职责边界。

未过滤：推荐=手机，logit=5.201，是否已购=True
已过滤：推荐=降噪耳机，logit=5.201，真实下次购买=降噪耳机
修复结论：模型负责相关性，候选层仍必须执行已购、库存与合规过滤。


## 生产差距

线上推荐需要多路召回、曝光未点负例、时间衰减、位置偏差校正、库存价格约束和在线特征服务。纯用户 ID embedding 无法处理新用户，因此要增加上下文或热门回退。离线 HitRate 还需扩展到 NDCG、覆盖率、多样性和长期价值，并通过 A/B 实验验证。

## 最小回归测试

In [7]:
assert len(users) >= 6 and len(items) >= 6  # 保证案例具有足够多的真实语义用户与商品。
assert loss_trace[-1] < loss_trace[0]  # 保证真实反向传播使排序损失下降。
assert first_gradient_norm > 0.0  # 保证用户 embedding 在首轮获得非零梯度。
assert neural_hit_rate > baseline_hit_rate  # 保证神经模型在同一候选口径上超过热门基线。
assert neural_hit_rate == 1.0  # 保证类目迁移在六位用户上找回全部留出商品。
assert raw_is_consumed  # 保证失败案例真实复现了已购商品泄漏。
assert filtered_top_index == held_out[example_user]  # 保证已购过滤后返回小林的真实下一次购买。
print("回归测试通过：训练更新、候选排序和已购过滤均符合预期。")  # 输出集中断言的最终验收结果。

回归测试通过：训练更新、候选排序和已购过滤均符合预期。
